This is our multi step algorithm to determine best locations for a coffee shop given our knowledge graph

1. Build a Node Regression Pipeline: 
    - Target Output: Avg_rating of a BusinessLocation Node
    - Input Features: Block Group attributes, Business Location Node locations, BusLoc - BusLoc relationships)

2. Extract Weights from Node Regression

3. Generate an "Optimal" Block Group

4. Similarity Search to Find Real Block Groups Closest to the Optimal Block Group
    - Rank by closeness in vector space 

5. Generate 10 New Sample Locations to Test
    - Use the geographical distribution of the businesses and county boundaries 
    - Test against zone locations to auto disqualify

6. Input the Generated Sample Locations into the Knowledge Graph

7. Use Node Regression Pipeline to Return Avg_Rating
    - Rank locations based on rating and other criteria


### 1. Setup

In [1]:
import pandas as pd

# import geopandas as gpd

import os

from dotenv import load_dotenv
from decimal import Decimal
from neo4j import GraphDatabase


In [2]:



group_driver = GraphDatabase.driver(
     "bolt://67.58.49.87:7687",
     auth=("neo4j", "h2u9l4px")
)


with group_driver.session() as session:
    result = session.run("MATCH (n) UNWIND labels(n) AS label RETURN count(DISTINCT label) AS count")
    num_nodes = result.single()["count"]
    print(f"Connection Successful: {num_nodes} unique node types found in the graph database")


Connection Successful: 11 unique node types found in the graph database


### 2. Best Location Algorithm

#### 2.1A Node Regression

##### 2.1.1. Delete Existing Graph Projections and Pipelines

In [277]:
delete_pipeline_query = """
CALL gds.pipeline.list() 
YIELD pipelineName
CALL gds.pipeline.drop(pipelineName) 
YIELD pipelineName AS droppedPipeline
RETURN 'Dropped pipeline: ' + droppedPipeline AS Result


"""

with group_driver.session() as session:
    result = session.run(delete_pipeline_query)
    for record in result:
        print(record)




<Record Result='Dropped pipeline: pipe-with-context'>


In [278]:


delete_graph_projection_query = """
CALL gds.graph.list() 
YIELD graphName
CALL gds.graph.drop(graphName) 
YIELD graphName AS droppedGraph
RETURN 'Dropped projected graph: ' + droppedGraph AS Result

"""

with group_driver.session() as session:
    result = session.run(delete_graph_projection_query)
    for record in result:
        print(record)




<Record Result='Dropped projected graph: bus_in_blockgroup_Graph'>


In [279]:





delete_models_query = """
CALL gds.model.list() 
YIELD modelName
CALL gds.model.drop(modelName) 
YIELD modelName AS droppedModel
RETURN 'Dropped GDS model: ' + droppedModel AS Result

"""

with group_driver.session() as session:
    result = session.run(delete_models_query)
    for record in result:
        print(record)




<Record Result='Dropped GDS model: nr-pipeline-model-contextual'>


##### 2.1.2. Configure Pipeline

In [280]:

create_graph_projection_query = """

MATCH (bl:BusinessLocation)
WHERE bl.avg_rating IS NOT NULL AND bl.avg_rating = bl.avg_rating // Filter out NaN/NULL
WITH bl, (bl.avg_rating - 1.0) / (5.0 - 1.0) AS calculated_scaled_rating, labels(bl) AS bl_labels

OPTIONAL MATCH (bl:BusinessLocation)-[r:contained_in]->(bg:BlockGroup) 
RETURN gds.graph.project(
  'bus_in_blockgroup_Graph',
  bl,
  bg,
  {
    sourceNodeLabels: ['bg'],
    targetNodeLabels: ['bl'],
    sourceNodeProperties: bg {.avghinc_cy, .totpop_cy, .crmcytotc},
    targetNodeProperties: bl {.avg_rating, .latitude, .longitude },
    relationshipType: 'contained_in'
  },
  { undirectedRelationshipTypes: ['contained_in'] }
)"""


with group_driver.session() as session:
    result = session.run(create_graph_projection_query)
    for record in result:
        print(record)



<Record gds.graph.project(
  'bus_in_blockgroup_Graph',
  bl,
  bg,
  {
    sourceNodeLabels: ['bg'],
    targetNodeLabels: ['bl'],
    sourceNodeProperties: bg {.avghinc_cy, .totpop_cy, .crmcytotc},
    targetNodeProperties: bl {.avg_rating, .latitude, .longitude },
    relationshipType: 'contained_in'
  },
  { undirectedRelationshipTypes: ['contained_in'] }
)={'graphName': 'bus_in_blockgroup_Graph', 'configuration': {'jobId': '956595c9-c4d4-4c25-9f7a-d923493f230c', 'query': "\n\nMATCH (bl:BusinessLocation)\nWHERE bl.avg_rating IS NOT NULL AND bl.avg_rating = bl.avg_rating // Filter out NaN/NULL\nWITH bl, (bl.avg_rating - 1.0) / (5.0 - 1.0) AS calculated_scaled_rating, labels(bl) AS bl_labels\n\nOPTIONAL MATCH (bl:BusinessLocation)-[r:contained_in]->(bg:BlockGroup) \nRETURN gds.graph.project(\n  'bus_in_blockgroup_Graph',\n  bl,\n  bg,\n  {\n    sourceNodeLabels: ['bg'],\n    targetNodeLabels: ['bl'],\n    sourceNodeProperties: bg {.avghinc_cy, .totpop_cy, .crmcytotc},\n    targetNode

In [281]:
create_pipe_w_context_query = """
CALL gds.alpha.pipeline.nodeRegression.create('pipe-with-context')"""

with group_driver.session() as session:
    result = session.run(create_pipe_w_context_query)
    for record in result:
        print(record)



<Record name='pipe-with-context' nodePropertySteps=[] featureProperties=[] splitConfig={'testFraction': 0.3, 'validationFolds': 3} autoTuningConfig={'maxTrials': 10} parameterSpace={'LinearRegression': [], 'RandomForest': []}>


In [282]:
add_bl_property_query = """


CALL gds.alpha.pipeline.nodeRegression.addNodeProperty('pipe-with-context', 'scaleProperties', {
    // These are the raw properties on the BusinessLocation node
    nodeProperties: ['latitude', 'longitude'], 
    scaler: 'MinMax',
    mutateProperty: 'scaled_predictors_local' 
}) 
YIELD name
"""



with group_driver.session() as session:

    result = session.run(add_bl_property_query)

    for record in result:

        print(record)





<Record name='pipe-with-context'>


In [283]:
# add_bg_property_query = """

# CALL gds.alpha.pipeline.nodeRegression.addNodeProperty('pipe-with-context', 'scaleProperties', {

#     nodeProperties: ['avghinc_cy', 'crmcytotc', 'totpop_cy'],
#     contextNodeLabels: ['bg'],
#     mutateProperty: 'context_data_scaled', 
#     scaler: 'MinMax'
# }) YIELD name AS stepName"""



# with group_driver.session() as session:

#     result = session.run(add_bg_property_query)

#     for record in result:

#         print(record)



In [284]:
add_node_query = """
CALL gds.alpha.pipeline.nodeRegression.addNodeProperty('pipe-with-context', 'fastRP', {
  embeddingDimension: 64,
  iterationWeights: [0, 1],
  mutateProperty:'embedding',
  contextNodeLabels: ['bg'],
  randomSeed: 1337
})"""

with group_driver.session() as session:
    result = session.run(add_node_query)
    for record in result:
        print(record)


<Record name='pipe-with-context' nodePropertySteps=[{'name': 'gds.scaleProperties.mutate', 'config': {'contextNodeLabels': [], 'mutateProperty': 'scaled_predictors_local', 'scaler': 'MinMax', 'contextRelationshipTypes': [], 'nodeProperties': ['latitude', 'longitude']}}, {'name': 'gds.fastRP.mutate', 'config': {'randomSeed': 1337, 'contextRelationshipTypes': [], 'iterationWeights': [0, 1], 'embeddingDimension': 64, 'contextNodeLabels': ['bg'], 'mutateProperty': 'embedding'}}] featureProperties=[] splitConfig={'testFraction': 0.3, 'validationFolds': 3} autoTuningConfig={'maxTrials': 10} parameterSpace={'LinearRegression': [], 'RandomForest': []}>


In [285]:
select_features_query = """ 
CALL gds.alpha.pipeline.nodeRegression.selectFeatures('pipe-with-context', [
    'embedding', 
    'scaled_predictors_local'//, 
//    'context_data_scaled'
]) YIELD name, featureProperties
"""

with group_driver.session() as session:
    result = session.run(select_features_query)
    for record in result:
        print(record)





<Record name='pipe-with-context' featureProperties=['embedding', 'scaled_predictors_local']>


In [286]:
split_query = """
CALL gds.alpha.pipeline.nodeRegression.configureSplit('pipe-with-context', {
  testFraction: 0.2,
  validationFolds: 5
}) YIELD splitConfig"""


with group_driver.session() as session:
    result = session.run(split_query)
    for record in result:
        print(record)


<Record splitConfig={'testFraction': 0.2, 'validationFolds': 5}>


##### 2.1.3. Model

In [287]:
# add_linear_model_query = """ 
# CALL gds.alpha.pipeline.nodeRegression.addLinearRegression('pipe-with-context', {
#     maxEpochs: 100, 
#     learningRate: 0.001,
#     tolerance: 0.001 
# })
# """
add_random_forest_model_query = """ 
CALL gds.alpha.pipeline.nodeRegression.addRandomForest('pipe-with-context', {numberOfDecisionTrees: 5})"""

with group_driver.session() as session:
    # result = session.run(add_linear_model_query)
    result = session.run(add_random_forest_model_query)

    for record in result:
        print(record)




<Record name='pipe-with-context' nodePropertySteps=[{'name': 'gds.scaleProperties.mutate', 'config': {'contextNodeLabels': [], 'mutateProperty': 'scaled_predictors_local', 'scaler': 'MinMax', 'contextRelationshipTypes': [], 'nodeProperties': ['latitude', 'longitude']}}, {'name': 'gds.fastRP.mutate', 'config': {'randomSeed': 1337, 'contextRelationshipTypes': [], 'iterationWeights': [0, 1], 'embeddingDimension': 64, 'contextNodeLabels': ['bg'], 'mutateProperty': 'embedding'}}] featureProperties=['embedding', 'scaled_predictors_local'] splitConfig={'testFraction': 0.2, 'validationFolds': 5} autoTuningConfig={'maxTrials': 10} parameterSpace={'LinearRegression': [], 'RandomForest': [{'maxDepth': 2147483647, 'minLeafSize': 1, 'minSplitSize': 2, 'numberOfDecisionTrees': 5, 'methodName': 'RandomForest', 'numberOfSamplesRatio': 1.0}]}>


In [ ]:
  

train_model_query = """ 
CALL gds.alpha.pipeline.nodeRegression.train('bus_in_blockgroup_Graph', {
  pipeline: 'pipe-with-context',
  targetNodeLabels: ['bl'],
  modelName: 'nr-pipeline-model-contextual',
  targetProperty: 'avg_rating',
  randomSeed: 25,
  concurrency: 1,
  metrics: ['MEAN_SQUARED_ERROR']
}) YIELD modelInfo
RETURN
  modelInfo.bestParameters AS winningModel,
  modelInfo.metrics.MEAN_SQUARED_ERROR.train.avg AS avgTrainScore,
  modelInfo.metrics.MEAN_SQUARED_ERROR.outerTrain AS outerTrainScore,
  modelInfo.metrics.MEAN_SQUARED_ERROR.test AS testScore
"""





with group_driver.session() as session:
    result = session.run(train_model_query)
    for record in result:
        print(record)




<Record winningModel={'maxDepth': 2147483647, 'minLeafSize': 1, 'minSplitSize': 2, 'numberOfDecisionTrees': 5, 'methodName': 'RandomForest', 'numberOfSamplesRatio': 1.0} avgTrainScore=0.5408671247115195 outerTrainScore=0.5018630872483216 testScore=0.4980480000000001>


##### Evaluate Model


In [289]:
# debug_query = """

# SHOW PROCEDURES 
# YIELD name, description
# WHERE name STARTS WITH 'gds.beta.pipeline.'
# RETURN name, description
# ORDER BY name"""


# """CALL gds.version()
# """
# """
# SHOW PROCEDURES
# YIELD name, description
# WHERE name CONTAINS 'property' AND name CONTAINS 'aggregate'
# RETURN name, description"""



# """SHOW PROCEDURES
# YIELD name, description
# WHERE name CONTAINS 'nodeProperty' OR name CONTAINS 'nodeProperties'
# RETURN name, description
# ORDER BY name
# """


# with group_driver.session() as session:
#     result = session.run(debug_query)
#     for record in result:
#         print(record)




In [290]:


  

get_variance_query = """ 
MATCH (bl:BusinessLocation)
WHERE bl.avg_rating IS NOT NULL 
  AND bl.avg_rating = bl.avg_rating // Exclude NaN values
RETURN 
    stDev(bl.avg_rating) AS StandardDeviation,
    stDev(bl.avg_rating)^2 AS Variance,
    avg(bl.avg_rating) AS MeanRating

"""

with group_driver.session() as session:
    result = session.run(get_variance_query)
    for record in result:
        print(record)




<Record StandardDeviation=0.6334417662994538 Variance=0.40124847129257185 MeanRating=4.312655732733582>


#### 2.1 B Node Classification

In [301]:
add_rating_class_query = """
MATCH (b:BusinessLocation)
WHERE b.avg_rating IS NOT NULL
SET b.rating_class = 
    CASE
        WHEN b.avg_rating >= 1 AND b.avg_rating < 3 THEN 'bad'
        WHEN b.avg_rating >= 3 AND b.avg_rating < 4 THEN 'okay'
        WHEN b.avg_rating >= 4 AND b.avg_rating < 4.5 THEN 'good'
        WHEN b.avg_rating >= 4.5 AND b.avg_rating < 5.0 THEN 'great'
        WHEN b.avg_rating = 5.0 THEN 'amazing'
        ELSE 'unrated_or_invalid' // Handles ratings outside the defined ranges
    END
RETURN count(b) AS updated_business_locations"""


with group_driver.session() as session:
    result = session.run(add_rating_class_query)
    for record in result:
        print(record)



<Record updated_business_locations=39593>


#### 2.2 Extract Model Weights

In [299]:






  

get_model_weights_query = """ 

CALL gds.model.list('nr-pipeline-model-contextual')
YIELD modelInfo
RETURN modelInfo
"""

# get_model_weights_query = """ 
# CALL gds.model.list('nr-pipeline-model-contextual')
# YIELD modelInfo
# UNWIND modelInfo.modelDetails.featureScores AS featureScore
# RETURN
#     featureScore.feature AS FeatureName,
#     featureScore.score AS ImportanceScore
# ORDER BY ImportanceScore DESC
# """

with group_driver.session() as session:
    result = session.run(get_model_weights_query)
    for record in result:
        print(record)






<Record modelInfo={'pipeline': {'featureProperties': [{'feature': 'embedding'}, {'feature': 'scaled_predictors_local'}], 'nodePropertySteps': [{'name': 'gds.scaleProperties.mutate', 'config': {'contextNodeLabels': [], 'mutateProperty': 'scaled_predictors_local', 'scaler': 'MinMax', 'contextRelationshipTypes': [], 'nodeProperties': ['latitude', 'longitude']}}, {'name': 'gds.fastRP.mutate', 'config': {'randomSeed': 1337, 'contextRelationshipTypes': [], 'iterationWeights': [0, 1], 'embeddingDimension': 64, 'contextNodeLabels': ['bg'], 'mutateProperty': 'embedding'}}]}, 'metrics': {'MEAN_SQUARED_ERROR': {'test': 0.4980480000000001, 'outerTrain': 0.5018630872483216, 'validation': {'min': 0.4679599999999999, 'avg': 0.6807294789915966, 'max': 0.8025882352941176}, 'train': {'min': 0.3947614255765196, 'avg': 0.5408671247115195, 'max': 0.7150113207547172}}}, 'featureProperties': ['embedding', 'scaled_predictors_local'], 'bestParameters': {'maxDepth': 2147483647, 'minLeafSize': 1, 'minSplitSize':

#### 2.3

#### 2.4

#### 2.5 Sample Business Location Generation

In [3]:
from shapely.geometry import Point
import geopandas as gpd

In [4]:
SD_crs = 'EPSG:26946'
common_crs = "EPSG:4326"


data_path = '../data/'



In [7]:

business_location_df = pd.read_json("../data/nodes/business_location.json")
business_location_geometry = [Point(xy) for xy in zip(business_location_df.longitude, business_location_df.latitude)]
business_location_gdf = gpd.GeoDataFrame(business_location_df, crs="EPSG:4326", geometry=business_location_geometry)
business_location_gdf = business_location_gdf[business_location_gdf['longitude']!=180] #fix these outliers


zone_location_df = pd.read_json("../data/nodes/zone_location.json")
zone_location_geometry = gpd.GeoSeries.from_wkt(zone_location_df['geom_ewkt'])
zone_location_gdf = gpd.GeoDataFrame(zone_location_df, crs="EPSG:4326", geometry = zone_location_geometry)

block_group_df = pd.read_json("../data/nodes/block_group.json")
block_group_geometry = gpd.GeoSeries.from_wkt(block_group_df['geom_wkt'])
block_group_gdf = gpd.GeoDataFrame(block_group_df, crs="EPSG:4326", geometry = block_group_geometry)





boundaries_zip = 'City_and_County_Boundaries.zip'
boundaries_path = os.path.join(data_path, boundaries_zip)
boundaries_gdf = gpd.read_file(boundaries_path)


boundaries_gdf = boundaries_gdf.to_crs(common_crs)

sd_boundaries_gdf = boundaries_gdf[boundaries_gdf['COUNTY_NAM'] == 'San Diego']
sd_county_geom = sd_boundaries_gdf.geometry.union_all()




In [ ]:
business_location_df['avg_rating'].hist(bins=200, legend=True)

In [ ]:
bl_sampled_points = sd_boundaries_gdf.sample_points(method='cluster_poisson',size = 10)

In [ ]:

m = sd_boundaries_gdf.explore()
bl_sampled_points.explore(m=m, color='red')

In [ ]:
bl_sampled_points.head()

### 2.6 Input New Samples to Knowledge Graph

### 2.7 Classify New Samples

In [302]:
classify_query = """
CALL gds.alpha.pipeline.nodeRegression.predict.stream('bus_in_blockgroup_Graph', {
  modelName: 'pipe-with-context',
  targetNodeLabels: ['NewBusinessLocation']
}) YIELD nodeId, predictedValue
WITH gds.util.asNode(nodeId) AS businessLocationNode, predictedValue AS predictedRatingClass
RETURN predictedRatingClass
  
ORDER BY predictedRatingClass"""

with group_driver.session() as session:
    result = session.run(classify_query)
    for record in result:
        print(record)


ClientError: {code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.alpha.pipeline.nodeRegression.predict.stream`: Caused by: java.util.NoSuchElementException: Model with name `pipe-with-context` does not exist.}